# CNN + Streamlit: Clasificacion de imagenes (CIFAR-10)
Notebook de experimentacion: EDA, preprocesamiento, entrenamiento, evaluacion y prediccion.

## 1. Instalacion de dependencias (Colab)

In [ ]:
!pip -q install tensorflow streamlit scikit-learn seaborn pillow
import tensorflow as tf
print(tf.__version__)

## 2. Imports y configuracion

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

CLASS_NAMES = ['avion','automovil','pajaro','gato','ciervo','perro','rana','caballo','barco','camion']
IMG_SIZE=(32,32)
NUM_CLASSES=10
SEED=42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 3. Carga del dataset

In [ ]:
(x_train,y_train),(x_test,y_test)=keras.datasets.cifar10.load_data()
y_train=y_train.flatten(); y_test=y_test.flatten()
print('Train',x_train.shape,y_train.shape)
print('Test',x_test.shape,y_test.shape)

## 4. Exploracion y visualizacion

In [ ]:
print('Min/Max pixeles:', x_train.min(), x_train.max())
uniq,cnts=np.unique(y_train,return_counts=True)
print(dict(zip(uniq.tolist(),cnts.tolist())))

plt.figure(figsize=(12,6))
for i in range(10):
    idx=np.where(y_train==i)[0][0]
    plt.subplot(2,5,i+1)
    plt.imshow(x_train[idx]); plt.title(CLASS_NAMES[i]); plt.axis('off')
plt.tight_layout(); plt.show()

## 5. Preprocesamiento y normalizacion

In [ ]:
x_train=x_train.astype('float32')/255.0
x_test=x_test.astype('float32')/255.0
print('Nuevo rango:', x_train.min(), x_train.max())

## 6. Division train/val/test

In [ ]:
x_tr,x_val,y_tr,y_val=train_test_split(x_train,y_train,test_size=0.1,random_state=SEED,stratify=y_train)
print('Train',x_tr.shape,'Val',x_val.shape,'Test',x_test.shape)

## 7. Data Augmentation + Construccion CNN

In [ ]:
augmentation=keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
],name='data_augmentation')

def build_cnn():
    inp=keras.Input(shape=(32,32,3))
    x=augmentation(inp)
    for filters in [32,64,128]:
        x=layers.Conv2D(filters,(3,3),padding='same',activation='relu')(x)
        x=layers.BatchNormalization()(x)
        x=layers.Conv2D(filters,(3,3),padding='same',activation='relu')(x)
        x=layers.BatchNormalization()(x)
        x=layers.MaxPooling2D((2,2))(x)
        x=layers.Dropout(0.25)(x)
    x=layers.Flatten()(x)
    x=layers.Dense(256,activation='relu')(x)
    x=layers.BatchNormalization()(x)
    x=layers.Dropout(0.5)(x)
    out=layers.Dense(NUM_CLASSES,activation='softmax')(x)
    m=keras.Model(inp,out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    return m

model=build_cnn()
model.summary()

## 8. Entrenamiento

In [ ]:
os.makedirs('models',exist_ok=True)
os.makedirs('outputs',exist_ok=True)
callbacks=[
  keras.callbacks.EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True),
  keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=2,min_lr=1e-5),
  keras.callbacks.ModelCheckpoint('models/cnn_cifar10.keras',monitor='val_accuracy',save_best_only=True,verbose=1),
]
history=model.fit(x_tr,y_tr,validation_data=(x_val,y_val),epochs=20,batch_size=64,callbacks=callbacks,verbose=1)
model.save('models/cnn_cifar10.keras')

## 9. Graficas del entrenamiento

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(history.history['loss'],label='train');ax[0].plot(history.history['val_loss'],label='val');ax[0].set_title('Loss');ax[0].legend()
ax[1].plot(history.history['accuracy'],label='train');ax[1].plot(history.history['val_accuracy'],label='val');ax[1].set_title('Accuracy');ax[1].legend()
plt.tight_layout(); plt.savefig('outputs/training_history.png',dpi=120); plt.show()

## 10. Evaluacion en test

In [ ]:
loss,acc=model.evaluate(x_test,y_test,verbose=0)
print(f'Test loss={loss:.4f} | Test accuracy={acc:.4f}')
probs=model.predict(x_test,verbose=0)
y_pred=np.argmax(probs,axis=1)
print(classification_report(y_test,y_pred,target_names=CLASS_NAMES))

## 11. Matriz de confusion

In [ ]:
cm=confusion_matrix(y_test,y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES)
plt.xlabel('Predicho');plt.ylabel('Real');plt.title('Matriz de confusion')
plt.tight_layout(); plt.savefig('outputs/confusion_matrix.png',dpi=120); plt.show()

## 12. Minimo 10 predicciones con imagenes

In [ ]:
idx=np.random.choice(len(x_test),size=10,replace=False)
plt.figure(figsize=(15,6))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(x_test[idx[i]])
    color='green' if y_test[idx[i]]==y_pred[idx[i]] else 'red'
    conf=np.max(probs[idx[i]])
    plt.title(f'R:{CLASS_NAMES[y_test[idx[i]]]}
P:{CLASS_NAMES[y_pred[idx[i]]]}
{conf:.2f}',color=color,fontsize=9)
    plt.axis('off')
plt.tight_layout(); plt.savefig('outputs/predictions.png',dpi=120); plt.show()

## 13. Guardado y carga del modelo

In [ ]:
m=tf.keras.models.load_model('models/cnn_cifar10.keras')
print('Modelo cargado OK. Input shape:',m.input_shape)

## 14. Prediccion sobre imagen subida (ejemplo)

In [ ]:
# Ejemplo de uso con PIL
# img=Image.open('mi_imagen.jpg').convert('RGB').resize(IMG_SIZE)
# arr=np.expand_dims(np.asarray(img,dtype='float32')/255.0,axis=0)
# p=m.predict(arr,verbose=0)[0]; i=int(np.argmax(p))
# print(CLASS_NAMES[i], float(p[i]))
print('Listo')